#### This plot shows the amount of mirror mode waves, normalized and amount of LAP data

### Normalization

### Functions

In [1]:
import datetime 
import numpy as np

import mpmath as mp
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import copy
from sklearn.preprocessing import StandardScaler
%matplotlib qt

from matplotlib.backends.backend_pdf import PdfPages
import os 
from os import listdir
from os.path import isfile, join
#------------------------------------------------------------------------------------
#This function receives a list with magnetic fields data and returns a list with the values of deltaB/B

def newdeltas(magneticfieldcolumn):
    rolling4=magneticfieldcolumn.rolling(600,center=True).mean() #moving average
    element=[] 
    for index1 in np.arange(0,len(rolling4)): #for every element in the rolling average
        if rolling4[index1]==np.nan:
            element.append(np.nan)
        else:    
            delta=((2*(magneticfieldcolumn[index1]-rolling4[index1]))/rolling4[index1])
            element.append(abs(delta))
            
    return element
#-----------------------------------------------------------------------------------------
#PCAnew returns the sorted eigenvalues and eigenvectors of the covariance matrix calculated with the X data 

def PCAnew(X):
    
    #The covariance matrix does not work with NaN values
    #------------------------------------------------
    check_for_nan_1 = X['Bx'].isnull()
    data_copy = X.copy()
    for i in np.arange(0,len(X)):
        if check_for_nan_1[i]==True:
            data_copy.drop(i, axis=0, inplace=True)
    #After this we droped all the nan values in the specific interval
    #-----------------------------------------------------------------------------
    if len(data_copy)!=1 and len(data_copy)!=2 and len(data_copy)!=3 and X.dropna().empty==False: #If the interval does not contain nan numbers
    
        x_std = StandardScaler().fit_transform(data_copy)
        #------------------------------------------------
    
        # features are columns from x_std
        features = x_std.T 
        covariance_matrix = np.cov(features)
        #print('Covariance matrix \n%s' %covariance_matrix)

        #Eigen Vectors and Eigen Values from Covariance Matrix
        eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
        #In order to print in descending order the vectors...
        vec=eig_vecs.T
        sortedd=sorted(eig_vals, reverse=True)
        neww=[]
    
        for index1 in np.arange(0,3):
             for index2 in np.arange(0,3):
                if eig_vals[index2]==sortedd[index1]:
                    neww.append(vec[index2])
    else:
        sortedd=[np.nan, np.nan, np.nan]
        neww=[[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan]]
        #print('Eigenvectors \n%s' %neww)
        #print('\nEigenvalues \n%s' %sortedd)
    return sortedd, neww  #In order to compute easier it is used the transpose of the eig_vecs 

#-------------------------------------------------------------------------------------------
#Angles between the vectors

import numpy as np
import numpy.linalg as LA

def angles(a,b): #Returns the angle in degrees and radians
    inner = np.inner(a, b)
    norms = LA.norm(a) * LA.norm(b)

    cos = inner / norms
    rad = np.arccos(np.clip(cos, -1.0, 1.0))
    deg = np.rad2deg(rad)
    
    return deg, rad

#-----------------------------------------------------------------------------------------
#bgmgvector is a list with the background magnetic fields vector
  
#The method consist of an iterating decomposing of the data matrix second by second
#For example if the time used is 3 minutes-->180 seconds. The method will use 90 seconds to the left and 90 seconds to the right (similar to the moving average)
#It should be noted that in this example the first 90 and last 90 points will not have any value 


#This function returns a list of lists that contains the eigenvalues ratios of the PCA method, 
#the angles phi and theta and the biggest deltaB/B value for each iteration

def function1(X,n, newdeltaB, bgmgvector): #n must be an even integer
    element=[]
    nan=[np.nan, np.nan, np.nan, np.nan, np.nan]
    L=X.shape[0] #Size of the data
    for i in np.arange(0,L): #For each data set 
        #The first two ifs are going to tell us if the interval exists and allows the iteration to work, based on the quantity of points 
        if i<(n/2):
            element.append(nan)
        elif (L-i)<=(n/2):
            element.append(nan) 
        else: #If the interval exists
            newdata=X.iloc[np.arange(i-n/2,i+1+n/2),:] #Select the interval
            newdataf=newdata.reset_index(drop=True) 
            newdata2=list(newdeltaB[i] for i in np.arange(int(i-n/2),int(i+1+n/2)))
            method=PCAnew(newdataf) #PCA using the data of the interval (this fuction returns the eigenvalues and eigenvectors)
            
            if np.isnan(method[0][0])==False and np.isnan(method[1][0][0])==False: #If the PCA exist
                eigenvector1=method[1][0] #eigenvector associated with the biggest eigenvalue 
                eigenvector2=method[1][2] #eigenvector associated with the smallest eigenvalue
                ratio1=method[0][0]/method[0][1] #Goodness betwen the biggest and the middle eigenvalue
                ratio2=method[0][1]/method[0][2] #Goodness betwen the middle and smallest eigenvalue
                mgvector=bgmgvector[i] #each background magnetic field vector
                theta=angles(eigenvector2,mgvector)[0] #angle in degrees associated with the smallest eigenvalue
                phi=angles(eigenvector1,mgvector)[0] #angle in degrees associated with the biggest eigenvalue
                #If the angle is needed in radians change angles[0] to angles [1]
                
                array=[ratio1, ratio2, theta,phi, max(newdata2)]  ##### MAYBE THIS IS NOT WORKING
                
                #array=[ratio1, ratio2, theta,phi, newdeltaB[i]]
                element.append(array)
            else:
                element.append(nan)
    return element

#------------------------------------------------------------------------------------------------------
def getting_day(data_plasma, year, month, day):
    
    time_plasma=pd.to_datetime(data_plasma['t_utc'])
    #year=2015
    mask = time_plasma.dt.year == int(year)
    include = data_plasma[mask]
    time_plasma2=pd.to_datetime(include['t_utc'])
    #month='06'
    mask2=time_plasma2.dt.month == int(month)
    include2 = include[mask2]
    time_plasma3=pd.to_datetime(include2['t_utc'])
    #day='07'
    mask3=time_plasma3.dt.day == int(day)
    include3=include2[mask3]

    time_plasma4=pd.to_datetime(include3['t_utc'])
    
  
    return time_plasma4, include3['npl']
#------------------------------------------------------------------------------------------------------
def get_directory(folder):
    files= [a for a in listdir(folder) if isfile(join(folder,a))]
    
    return files

#------------------------------------------------------------------------------------------------------
#This function receives a Mirror Mode Waves Candidates DataFrame and returns its time intervals and how many MM are in that day

def intervals(element, n, array1):  
    #modifying the MM table
    element['tvalue'] = element.index
    element['delta'] = (element['tvalue']-element['tvalue'].shift()).fillna(0)
    
    zx=element.copy()
    zx2=zx.reset_index(drop=True)
    
    deltas=zx2['delta']
    deltas=deltas.values.tolist()
    
    indexA=zx2['Index']
    
    #Empty Dataframe
    my_df=pd.DataFrame()
    my_df2=pd.DataFrame()
    
    if len(element['tvalue'])!=0: #If we have MM waves
    
        #LIMIT CONDITIONS
        limits=[]
        limits2=[]
        for i in np.arange(0,len(zx2)):
            if deltas[i]>(n/2) or deltas[i]==0: #Same event
                limits.append(indexA[i])
                if i!=0:
                    limits2.append(indexA[i-1])
        limits2.append(indexA[len(deltas)-1])   
        
    
        if len(limits)!=1: #If there is not only one MM
    
            for i in np.arange(0,len(limits)): 
                index1=limits[i]
                index2=limits2[i]
                my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True) 
    
        else: #If there is only one MM
            index1=limits[0]
            index2=limits2[0]
            large=len(deltas)
            my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True)
    else: #If we have not MM
        my_df=my_df
        limits=[]
        limits2=[]


    return my_df, len(my_df), limits,limits2,

#---------------------------------------------------------------------------------------------------------------

In [4]:
#MM waves folder

MM_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/peakswithoutnegativevaluesnpl_CHECKED/peaks_201601"
month='January'
#Plasma folder
plasma_folder="C:/Users/Ariel/Desktop/ESA/data_LAP/2016/enero" 
año=2016
#---------------------------------------------------------

list_of_files_MM=get_directory(MM_folder)
#List with the paths
newlist2=[]
for item in list_of_files_MM:
    newlist2.append(MM_folder+'/'+str(item))

print(newlist2)
list_of_MM=[] #Amount of MM waves per day
for i in np.arange(0, len(newlist2)):
        title2=['Beginning','End','Index1','Index2','Pearson']
        path= str(newlist2[i])
        data1= pd.read_table(path, header=None, names=title2, sep='\t+')
        #data1= pd.read_table(path)
        data2=data1.copy()
        data2=data2.iloc[np.arange(1,len(data2)),:] #delete the first row
        #data2=data2.reset_index(drop=True)
        list_of_MM.append(len(data2))


['C:/Users/Ariel/Desktop/JUPYTER/Resultados/peakswithoutnegativevaluesnpl_CHECKED/peaks_201601/MM_final_table_with_pearson_prominence_method_20160101', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/peakswithoutnegativevaluesnpl_CHECKED/peaks_201601/MM_final_table_with_pearson_prominence_method_20160102', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/peakswithoutnegativevaluesnpl_CHECKED/peaks_201601/MM_final_table_with_pearson_prominence_method_20160103', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/peakswithoutnegativevaluesnpl_CHECKED/peaks_201601/MM_final_table_with_pearson_prominence_method_20160104', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/peakswithoutnegativevaluesnpl_CHECKED/peaks_201601/MM_final_table_with_pearson_prominence_method_20160105', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/peakswithoutnegativevaluesnpl_CHECKED/peaks_201601/MM_final_table_with_pearson_prominence_method_20160106', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/peakswithoutnegativevaluesnpl_CHECKED/peaks_201601

C:\Users\Ariel\anaconda3\lib\site-packages\pandas\io\parsers.py:765: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  return read_csv(**locals())


In [ ]:
print(list_of_MM)
print(len(list_of_MM))

In [5]:

#---------------------------------------------------------
list_of_files_plasma=get_directory(plasma_folder)
#List with the paths
newlist=[]
for item in list_of_files_plasma:
    newlist.append(plasma_folder+'/'+str(item))
print(newlist)  

day_plasma=[]
list_of_events=[]
for i in np.arange(0, len(newlist)):
        title2=['t_utc','?','npl','??','???','????']
        path= str(newlist[i])
        data1= pd.read_table(path, header=None, names=title2, sep=',', engine='python', parse_dates=['t_utc'])
        data2=data1.copy()
        data2=data2.iloc[np.arange(1,len(data2)),:] #delete the first row
        data2=data2.reset_index(drop=True)
        data2=data2[['t_utc','npl']]
        
        #--------------------------------------------------------------------------
        #Saving the dates
        path_time= pd.to_datetime(data2['t_utc']) #data2['t_utc']
        #It is needed to obtain the year, month and day of an specific path
        qqq=path_time.dt.day
        day=qqq[0]
        day_plasma.append(day)
        #------------------------------------------------------------------------------
        #Resample
        data2['index'] = pd.to_datetime(data2['t_utc'])
        data2.set_index('index', inplace=True)
        data2=data2.resample('1S').mean()

        data2['t_utc'] = pd.to_datetime(data2.index.values)
        data2= data2.reset_index()
        
        list_of_events.append([day,len(data2)])

#------------------------------------------------------        
final_list=[]
for i in np.arange(1,len(list_of_MM)+1):
    final_list.append([i,0])
#print(final_list)    


for i in np.arange(1,len(list_of_MM)+1):
    for j in np.arange(0,len(list_of_events)):
        if i==list_of_events[j][0]:
            final_list[list_of_events[j][0]-1]=[list_of_events[j][0],list_of_events[j][1]]
    
#print(final_list)    

for i in np.arange(0,len(list_of_MM)):
    final_list[i].append(list_of_MM[i])
    
print(final_list)          

['C:/Users/Ariel/Desktop/ESA/data_LAP/2016/enero/LAP_20160101_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2016/enero/LAP_20160102_000000_914_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2016/enero/LAP_20160103_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2016/enero/LAP_20160104_000000_914_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2016/enero/LAP_20160105_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2016/enero/LAP_20160106_000000_914_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2016/enero/LAP_20160107_000000_914_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2016/enero/LAP_20160108_000000_914_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2016/enero/LAP_20160109_000000_914_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2016/enero/LAP_20160110_000000_914_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2016/enero/LAP_20160111_000000_914_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2016/enero/LAP_20160112_000000_914_NEL.TAB', 'C:/Users/Ariel/Desktop/ESA/data_LAP/2016/enero/

## LINE PLOTS

In [6]:
import math

normalized=[]
normalized_log=[]

for i in np.arange(0,len(list_of_MM)):
    print(i)
    if final_list[i][2]==0 and final_list[i][1]==0:
        normalized.append(np.nan)
        normalized_log.append(np.nan)
    
    elif final_list[i][2]==0 and final_list[i][1]!=0:
        normalized.append(0)
        normalized_log.append(0)
    
    else:
        n=final_list[i][2]/final_list[i][1]
        normalized.append(n)
        normalized_log.append(-1*math.log(n,10))
        
print(normalized)   
print('sas')
print(normalized_log)
x=np.arange(1,len(list_of_MM)+1)



fig, ax = plt.subplots()
ax.plot(x,normalized_log,'r-',label=r'$\alpha$')
plt.xlabel('Days',fontsize=10)
plt.ylabel('-log('+r'$\alpha$'+')',fontsize=10)
plt.title('Mirror mode waves February 2015',fontsize=15)
plt.legend()
ax.set_xticks(x)
plt.grid()

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
[0, 0, 0, 0, 0, 0, 0, 2.6881720430107527e-05, 3.472222222222222e-05, 1.1574074074074073e-05, 5.2124055251498564e-05, 1.2087659708203894e-05, 9.910802775024777e-05, 3.530242409978819e-05, 5.372011818426001e-05, 5.787237982800329e-05, 2.3269071913066746e-05, 6.21048578419804e-05, 1.2664640324214791e-05, 0, 5.876821814762576e-05, 0, 1.2970168612191959e-05, 3.5842293906810036e-05, 4.6554934823091246e-05, 0, 3.41180484476288e-05, 0, 5.3763440860215054e-05, 1.511030522816561e-05, 8.363201911589008e-05]
sas
[0, 0, 0, 0, 0, 0, 0, 4.570542939881896, 4.459392487759231, 4.936513742478893, 4.282961803534335, 4.917657774755192, 4.003891166236911, 4.452195472033832, 4.2698630405544105, 4.237528658211557, 4.633220938203249, 4.206874427980389, 4.8974071396615795, 0, 4.2308574768946725, 0, 4.887054378050956, 4.445604203273597, 4.3320342770275175, 0, 4.467015818438434, 0, 4.269512944217916, 4.820726762823834, 4.0776274179

## BAR PLOTS

In [7]:
from matplotlib import style

#----------------------------
normalized2=[]
for i in np.arange(0,len(normalized)):
    normalized2.append(100000*normalized[i])
#print(normalized2)

#----------------------------
data_plasma=[]
for i in np.arange(0,len(final_list)):
    data_plasma.append(final_list[i][1])
#print(data_plasma)    

data_plasma2=[]
for i in np.arange(0,len(data_plasma)):
    data_plasma2.append(0.0001*data_plasma[i])
print(data_plasma2)
x=np.arange(1,len(list_of_MM)+1)

#style.use('ggplot')
plt.figure()
barWidth=0.2
barWidthnormalized=0.4
plt.bar(x-0.3,list_of_MM,color='royalblue', width=barWidth, label='Amount of Mirror Mode Waves')
plt.bar(x,normalized2,color='red', width=barWidthnormalized, label='Normalized Data')
plt.bar(x+0.3,data_plasma2,color='green', width=barWidth, label='Amount of LAP data')    

plt.xlabel('Days',fontsize=10)
plt.title('Mirror mode waves February 2015',fontsize=15)
plt.legend()

[6.672000000000001, 7.71, 8.370000000000001, 6.6240000000000006, 6.534000000000001, 6.336, 8.508000000000001, 7.44, 8.64, 8.64, 7.674, 8.2729, 6.054, 8.498000000000001, 7.446000000000001, 8.639700000000001, 8.5951, 8.0509, 7.896000000000001, 6.264, 8.508000000000001, 4.542, 7.71, 8.370000000000001, 6.444, 5.16, 5.862, 8.43, 5.58, 6.618, 8.370000000000001]


In [ ]:
#for i in np.arange(0,1):
 #   with PdfPages(r'C:/Users/Ariel/Desktop/JUPYTER/'+'Mirror_Mode_Waves_'+str(month)+'_'+str(año)+'.pdf') as export_pdf:
        
  #      plt.figure()
   #     barWidth=0.2
    #    barWidthnormalized=0.4
     #   plt.bar(x-0.3,list_of_MM,color='royalblue', width=barWidth, label='Amount of Mirror Mode Waves')
      #  plt.bar(x,normalized2,color='red', width=barWidthnormalized, label='Normalized Data')
       # plt.bar(x+0.3,data_plasma2,color='green', width=barWidth, label='Amount of LAP data')    

        #plt.xlabel('Days',fontsize=10)
        #plt.title('Mirror mode waves '+ str(month) + str(año),fontsize=15)
        #plt.legend() 
        
        #export_pdf.savefig()
        #plt.close()

### SAVING FILES

In [ ]:
import csv

#Creating a new dataframe in order to establish the anticorrelation
a2={ 'Days': np.arange(1,len(list_of_MM)+1), 'Normalized_Data': normalized ,'Data_Plasma': data_plasma, 'Waves': list_of_MM}
data_per_month= pd.DataFrame(a2,columns = ['Days', 'Normalized_Data','Data_Plasma','Waves'])
data_per_month.to_csv('Data'+'_'+str(month)+'_'+str(año), index=False, sep='\t')


In [ ]:
sas

### ENTIRE YEAR PLOT

In [9]:
#MM waves folder

#MM_folder="C:/Users/Ariel/Desktop/JUPYTER/Mirror_Mode_Waves_Linear_Regression/DATA"
MM_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/Mirror Mode Waves Per Month/Mirror_Mode_Waves_Using_Prominence_method_notnegativevalues_CHECKED"
#---------------------------------------------------------

list_of_files_MM=get_directory(MM_folder)

#List with the paths
newlist2=[]
for item in list_of_files_MM:
    newlist2.append(MM_folder+'/'+str(item))
print(newlist2)
events=[] #Amount of MM waves per day
for i in np.arange(0, len(newlist2)):
        title2=['Days','Normalized_Data','Data_Plasma','Waves']
        path= str(newlist2[i])
        data1= pd.read_table(path, header=None, names=title2, sep='\t', engine='python')
        #data1= pd.read_table(path)
        data2=data1.copy()
        data2=data2.iloc[np.arange(1,len(data2)),:] #delete the first row
        #data2=data2.reset_index(drop=True)
        events.append(data2)


['C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/Mirror Mode Waves Per Month/Mirror_Mode_Waves_Using_Prominence_method_notnegativevalues_CHECKED/Data_201411', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/Mirror Mode Waves Per Month/Mirror_Mode_Waves_Using_Prominence_method_notnegativevalues_CHECKED/Data_201412', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/Mirror Mode Waves Per Month/Mirror_Mode_Waves_Using_Prominence_method_notnegativevalues_CHECKED/Data_201501', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/Mirror Mode Waves Per Month/Mirror_Mode_Waves_Using_Prominence_method_notnegativevalues_CHECKED/Data_201502', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/Mirror Mode Waves Per Month/Mirror_Mode_Waves_Using_Prominence_method_notnegativevalues_CHECKED/Data_201503', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/Mirror Mode Waves Per Month/Mirror_Mode_Waves_Using_Prominence_method_notnegativevalues_CHECKED/Data_201504', 'C:/Users/Ariel

In [10]:
list_of_normalized_data=[]
list_of_data_plasma=[]
list_of_waves=[]
for i in np.arange(0,len(newlist2)):
    for j in np.arange(0,len(events[i])):
        list_of_normalized_data.append(events[i]['Normalized_Data'][j+1])
        list_of_data_plasma.append(events[i]['Data_Plasma'][j+1])
        list_of_waves.append(events[i]['Waves'][j+1])
     

In [11]:
#TOTAL EVENTS

#print(list_of_waves)
a=list_of_waves.copy()
aa=[]
for item in a:
    if type(item)==str:
        aa.append(int(item))
        
#print(aa)     
#print(sum(aa))

In [ ]:
print(type(list_of_waves[13]))

In [12]:
from matplotlib import style

#fix bug
#----------------------------
normalized2=[]
for i in np.arange(0,len(list_of_normalized_data)):
    normalized2.append(float(list_of_normalized_data[i]))
#print(normalized2)

#----------------------------
data_plasma=list_of_data_plasma.copy()
data_plasma_ses=[]    
for i in np.arange(0,len(data_plasma)):
    if type(data_plasma[i])!=str:
        data_plasma_ses.append(0)
    elif type(data_plasma[i])==str:
        data_plasma_ses.append(float(data_plasma[i]))

data_plasma2=[]
for i in np.arange(0,len(data_plasma)):
    data_plasma2.append(data_plasma_ses[i])
#print(data_plasma2)
#------------------------------
list_of_MM=[]
for i in np.arange(0,len(list_of_waves)):
    if type(list_of_waves[i])!=str:
        list_of_MM.append(0)
    elif type(list_of_waves[i])==str:
        list_of_MM.append(float(list_of_waves[i]))


print(list_of_MM)


x=np.arange(1,len(list_of_data_plasma)+1)
plt.figure()
barWidth=0.2
barWidthnormalized=0.8
plt.bar(x-0.5,list_of_MM,color='royalblue', width=barWidth, label='Amount of Mirror Mode Waves')
plt.bar(x,normalized2,color='red', width=barWidthnormalized, label='Normalized Data')
plt.bar(x+0.5,data_plasma2,color='green', width=barWidth, label='Amount of LAP data')    

plt.xlabel('Days',fontsize=10)
plt.title('Mirror mode waves 2015',fontsize=15)
plt.legend()

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.0, 0.0, 0.0, 0.0, 10.0, 0.0, 3.0, 1.0, 6.0, 0.0, 2.0, 0.0, 7.0, 0.0, 5.0, 1.0, 1.0, 6.0, 2.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 5.0, 4.0, 2.0, 6.0, 0.0, 6.0, 0.0, 12.0, 3.0, 2.0, 0.0, 4.0, 1.0, 10.0, 4.0, 0.0, 4.0, 2.0, 1.0, 1.0, 6.0, 4.0, 0.0, 4.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 6.0, 0.0, 0.0, 2.0, 3.0, 8.0, 0.0, 6.0, 0.0, 0.0, 1.0, 0.0, 0.0, 2.0, 0.0, 0.0, 2.0, 0.0, 11.0, 13.0, 2.0, 10.0, 6.0, 4.0, 4.0, 9.0, 0.0, 0.0, 0.0, 1.0, 4.0, 4.0, 3.0

## MM WAVES PLOT

In [16]:
#FIRST PLOT
plt.figure()
ax7=plt.subplot(313)
figsize=(20,4)
barWidth=0.3
barWidthnormalized=0.3
plt.bar(x,normalized2,color='red', width=barWidthnormalized, label='Normalized Data')   
#plt.xlabel('Days',fontsize=10)
plt.legend()

#specify x-axis locations
x_ticks = [1,31,62,90,121,151,182,212,243,274,304,335,365,396,427,456]

#specify x-axis labels
#x_labels = ['2014-Nov', '2014-Dec','2015-Jan','2015-Feb','2015-Mar','2015-Apr','2015-May','2015-Jun','2015-Jul','2015-Aug','2015-Sep','2015-Oct','2015-Nov','2015-Dec','2016-Jan','2016-Feb'] 
x_labels = ['2014-Nov', '','2015-Jan','','2015-Mar','','2015-May','','2015-Jul','','2015-Sep','','2015-Nov','','2016-Jan',''] 
#add x-axis values to plot
#plt.xticks(ticks=x_ticks, labels=x_labels,rotation=90,fontsize=8)
plt.xticks(ticks=x_ticks, labels=x_labels,fontsize=13)


#SECOND PLOT

ax1=plt.subplot(311,sharex=ax7)
figsize=(20,4)
barWidth=0.3
barWidthnormalized=0.3
plt.bar(x,data_plasma2,color='green', width=barWidth, label='Amount of LAP data')    
plt.setp(ax1.get_xticklabels(), fontsize=6)
plt.title('Mirror mode waves',fontsize=15)
plt.legend()
plt.setp(ax1.get_xticklabels(), visible=False)

#THIRD PLOT

ax2=plt.subplot(312,sharex=ax7)
figsize=(20,4)
barWidth=0.3
barWidthnormalized=0.3
plt.bar(x,list_of_MM, color='blue', width=barWidth, label='Amount of Mirror Mode Waves')    
#plt.setp(ax2.get_xticklabels(), fontsize=6)
#plt.title('Mirror mode waves 2015',fontsize=15)
#ax2.set_ylim([0,50]) #-10 and 10 
plt.legend()
plt.setp(ax2.get_xticklabels(), visible=False)

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import datetime

# dummy data (Days)
dates_d = pd.date_range('2010-01-01', '2013-12-31', freq='D')
df_year = pd.DataFrame(np.random.randint(100, 200, (dates_d.shape[0], 1)), columns=['Data'])
df_year.index = dates_d #set index

pt = pd.pivot_table(df_year, index=df_year.index.month, columns=df_year.index.year, aggfunc='sum')
pt.columns = pt.columns.droplevel() # remove the double header (0) as pivot creates a multiindex.

ax = plt.figure().add_subplot(111)
ax.plot(pt)

ticklabels = [datetime.date(1900, item, 1).strftime('%b') for item in pt.index]
ax.set_xticks(np.arange(1,13))
ax.set_xticklabels(ticklabels) #add monthlabels to the xaxis

ax.legend(pt.columns.tolist(), loc='center left', bbox_to_anchor=(1, .5)) #add the column names as legend.
plt.tight_layout(rect=[0, 0, 0.85, 1])

plt.show()

In [15]:



ax.set_xticks(np.arange(1,13))
#ax.set_xticklabels(ticklabels) #add monthlabels to the xaxis

# New plot

### Outgassing